# California Housing — HUGO MANZANO 36231

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

Cargamos el dataset y solo partimos 80/10/10

In [9]:
data = fetch_california_housing()
X, y = data.data, data.target

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)


Las features 0–5 se clipean con IQR y se escalan y la Latitude y Longitude (índices 6 y 7) se dejan asi como estam  El fit solo usa datos de train para no filtrarse a val/test

In [10]:
class Preprocessing(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('low',  torch.zeros(6))
        self.register_buffer('high', torch.zeros(6))
        self.register_buffer('mean', torch.zeros(6))
        self.register_buffer('std',  torch.ones(6))

    #Esta parte de la implementacion para clippear si es meramente AI, pero por mi parte si me quedó clara, se busca obtener un rango entre 25 y 75 para considerar outliers lo que este fuera
    #entonces despues usamos el clamp para ponerlo en el limite con el high y el low dependiendo el caso
    #con esto me quedó claro que se busca no perder datos pero tampoco esos datos que no quise perder vabn a modificar o distorsionar resultados
    def fit(self, X):
        num = X[:, :6]
        Q1, Q3 = torch.quantile(num, 0.25, dim=0), torch.quantile(num, 0.75, dim=0)
        IQR = Q3 - Q1
        self.low  = Q1 - 1.5 * IQR
        self.high = Q3 + 1.5 * IQR
        clipped   = num.clamp(self.low, self.high)
        self.mean = clipped.mean(0)
        self.std  = clipped.std(0).clamp(min=1e-8)

    def forward(self, x):
        num = (x[:, :6].clamp(self.low, self.high) - self.mean) / self.std
        return torch.cat([num, x[:, 6:]], dim=1)

Feed-forward simple, el preprocessing va al inicio

In [11]:
class HousingNet(nn.Module):
    def __init__(self, hidden_sizes, dropout=0.0):
        super().__init__()
        self.prep = Preprocessing()
        self.prep.fit(X_train_t)

        layers, in_size = [], 8
        for h in hidden_sizes:
            layers += [nn.Linear(in_size, h), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_size = h
        layers.append(nn.Linear(in_size, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(self.prep(x))

Aqui entrenamos entonces probamos 3 modelos con diferentes hiperparámetros y guardamos train y val loss en cada época

In [12]:
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
loss_fn = nn.MSELoss()

def train_model(model, lr, epochs=100):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_b, y_b in train_loader:
            pred = model(X_b)
            loss = loss_fn(pred, y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_loader))

        model.eval()
        with torch.no_grad():
            val_losses.append(loss_fn(model(X_val_t), y_val_t).item())

    return train_losses, val_losses

model1 = HousingNet([64])
model2 = HousingNet([128, 64])
model3 = HousingNet([256, 128, 64], dropout=0.2)

EPOCHS = 100
print("Entrenamos los modelos")
print("Modelo 1")
l1 = train_model(model1, lr=0.01,  epochs=EPOCHS)
print("Modelo 2")
l2 = train_model(model2, lr=0.001, epochs=EPOCHS)
print("Modelo 3")
l3 = train_model(model3, lr=0.001, epochs=EPOCHS)

Entrenamos los modelos
Modelo 1
Modelo 2
Modelo 3


Aqui evaluamos el mejor modelo

In [13]:
# Modelo 2 tuvo el menor val loss y lo evaluamos en test
best_model = model2
best_model.eval()
with torch.no_grad():
    test_mse  = loss_fn(best_model(X_test_t), y_test_t).item()
    test_rmse = test_mse ** 0.5

print(f"Test MSE:  {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Error promedio ~${test_rmse * 100_000:,.0f} USD")

Test MSE:  0.3969
Test RMSE: 0.6300
Error promedio ~$62,998 USD
